# 04 CV Quality Analysis

This notebook analyzes the quality of a CV independently from any specific job posting.

Main goals:
- load extracted CV text from the previous PDF extraction step
- use an LLM to analyze CV quality
- evaluate clarity, structure, completeness and IT relevance
- generate CV improvement recommendations
- calculate a transparent CV Quality Score
- save the result as JSON and Markdown report

This step does not compare the CV with a job posting.

## 1. Imports and Settings

In [1]:
import os
import json
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [2]:
load_dotenv(dotenv_path=Path("../.env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
PROCESSED_TEST_CV_DIR = Path("../data/processed/test_cv")
OUTPUT_CV_QUALITY_DIR = Path("../outputs/cv_quality")

cv_text_path = PROCESSED_TEST_CV_DIR / "test_cv_extracted_text.txt"

json_output_path = OUTPUT_CV_QUALITY_DIR / "cv_quality_analysis.json"
markdown_output_path = OUTPUT_CV_QUALITY_DIR / "cv_quality_report.md"

cv_text_path

WindowsPath('../data/processed/test_cv/test_cv_extracted_text.txt')

## 2. Load Extracted CV Text

In [4]:
with open(cv_text_path, "r", encoding="utf-8") as file:
    cv_text = file.read()

## 3. Basic Text Check

In [5]:
cv_text_stats = {
    "character_count": len(cv_text),
    "word_count": len(cv_text.split()),
    "line_count": len(cv_text.splitlines())
}


In [6]:
cv_text_stats

{'character_count': 2443, 'word_count': 327, 'line_count': 45}

## 4. Define Structured Output Schema

In [7]:
class CVQualityScores(BaseModel):

    structure_and_readability_score: int = Field(
        ge=0,
        le=100,
        description=(
            "Score from 0 to 100 for CV structure, readability and section organization. "
            "0-20: very poorly structured, hard to read, no clear sections. "
            "21-40: weak structure, inconsistent formatting, important sections difficult to identify. "
            "41-60: acceptable structure but with noticeable readability or organization issues. "
            "61-80: well structured and mostly easy to read, with minor formatting or organization issues. "
            "81-100: very clear, professional, logically organized and easy to scan."
        )
    )

    completeness_score: int = Field(
        ge=0,
        le=100,
        description=(
            "Score from 0 to 100 for completeness of basic CV information. "
            "Evaluate whether the CV includes relevant contact information, education, work experience, "
            "skills, projects, certifications or other important sections. "
            "0-20: most essential information is missing. "
            "21-40: several important sections are missing or unclear. "
            "41-60: basic information is present but incomplete. "
            "61-80: most important information is included. "
            "81-100: CV is complete and contains all key information expected for an IT candidate."
        )
    )

    technical_skills_clarity_score: int = Field(
        ge=0,
        le=100,
        description=(
            "Score from 0 to 100 for clarity and organization of technical skills. "
            "Evaluate whether programming languages, frameworks, databases, tools and platforms are clearly listed "
            "and grouped in a readable way. "
            "0-20: technical skills are missing or almost impossible to identify. "
            "21-40: skills are mentioned vaguely or mixed with unrelated text. "
            "41-60: skills are present but poorly organized or too generic. "
            "61-80: skills are clear and mostly well organized. "
            "81-100: skills are clearly structured, specific and highly readable."
        )
    )

    experience_description_score: int = Field(
        ge=0,
        le=100,
        description=(
            "Score from 0 to 100 for clarity and detail of work experience descriptions. "
            "Evaluate whether job roles, responsibilities, technologies used and contributions are clearly described. "
            "0-20: work experience is missing or not described. "
            "21-40: experience is listed but with very little detail. "
            "41-60: responsibilities are partially described but remain generic. "
            "61-80: experience is clearly described with relevant responsibilities and technologies. "
            "81-100: experience is detailed, specific and clearly shows candidate contribution."
        )
    )

    projects_description_score: int = Field(
        ge=0,
        le=100,
        description=(
            "Score from 0 to 100 for clarity and detail of project descriptions. "
            "Evaluate whether projects include goal, role, technologies, implementation details and outcome. "
            "0-20: projects are missing or not described. "
            "21-40: projects are listed with minimal explanation. "
            "41-60: projects are understandable but lack technical or outcome details. "
            "61-80: projects are clearly described with technologies and candidate role. "
            "81-100: projects are detailed, technically clear and include concrete outcomes or impact."
        )
    )

    measurable_results_score: int = Field(
        ge=0,
        le=100,
        description=(
            "Score from 0 to 100 for presence of measurable achievements and concrete results. "
            "Look for numbers, percentages, performance improvements, reduced costs, increased efficiency, "
            "number of users, project scale or other measurable outcomes. "
            "0-20: no measurable results are present. "
            "21-40: very few vague achievements are mentioned. "
            "41-60: some results are present but mostly not quantified. "
            "61-80: several concrete results or achievements are included. "
            "81-100: CV strongly uses quantified achievements and measurable impact."
        )
    )

    it_relevance_score: int = Field(
        ge=0,
        le=100,
        description=(
            "Score from 0 to 100 for relevance of the CV to IT roles. "
            "Evaluate how strongly the CV matches IT-related positions based on education, experience, projects, "
            "technical skills, tools and technologies. "
            "0-20: CV is not relevant to IT roles. "
            "21-40: weak IT relevance with only limited technical content. "
            "41-60: partially relevant to IT but missing important technical evidence. "
            "61-80: clearly relevant to IT roles. "
            "81-100: highly relevant to IT roles with strong technical background, projects and skills."
        )
    )

In [8]:
class CVQualityAnalysis(BaseModel):

    overall_summary: str = Field(
        description="Short summary of the overall CV quality."
    )

    scores: CVQualityScores = Field(
        description="Individual CV quality scores."
    )

    strengths: List[str] = Field(
        description="Strong aspects of the CV that are clearly visible from the provided text."
    )

    weaknesses: List[str] = Field(
        description="Weak aspects of the CV based only on the provided text."
    )

    missing_or_unclear_sections: List[str] = Field(
        description="Sections or information that are missing, unclear or not detailed enough."
    )

    cv_improvement_recommendations: List[str] = Field(
        description="Practical and honest recommendations for improving CV clarity, structure and completeness."
    )

    evidence_notes: List[str] = Field(
        description="Short notes explaining which parts of the CV text support the analysis."
    )

## 5. Initialize OpenAI Model

In [9]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


In [10]:
structured_llm = llm.with_structured_output(CVQualityAnalysis)

## 6. Create CV Quality Analysis Prompt

In [11]:
cv_quality_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI assistant specialized in CV quality analysis for IT job candidates.

Your task is to analyze the quality of the provided CV independently from any specific job posting.

Important rules:
- Analyze only the information present in the CV text.
- Do not invent skills, experience, education, certifications, projects or achievements.
- Do not assume that the candidate has a skill if it is not visible in the CV.
- If something is unclear or only partially presented, explicitly say that it is unclear or partially presented.
- Focus on clarity, structure, completeness, technical skills presentation, experience descriptions, project descriptions and IT relevance.
- Recommendations should improve the CV honestly, without suggesting false information.
- The analysis should be useful for an IT candidate.

Scoring rules:
- Assign all scores as integers from 0 to 100.
- Strictly follow the scoring rubrics defined in the CVQualityScores schema.
- Use the full 0-100 scale consistently across different CVs.
- Do not give high scores when the relevant information is missing, vague, generic or unsupported by the CV text.
- A score above 80 should be given only when that criterion is clearly strong and supported by specific evidence from the CV.
- A score between 60 and 80 means that the criterion is mostly satisfactory, but there are still minor issues or missing details.
- A score between 40 and 60 means that the criterion is partially satisfied, but important information is incomplete, unclear or too generic.
- A score below 40 means that the criterion is weak, mostly missing or poorly presented.
- Every score should be consistent with the strengths, weaknesses, missing_or_unclear_sections and evidence_notes fields.
- Evidence notes should briefly explain which parts of the CV support the assigned scores.

Return the analysis using the required structured schema.
"""
        ),
        (
            "human",
            """
Analyze the following CV text:

{cv_text}
"""
        )
    ]
)

## 7. Run CV Quality Analysis

In [12]:
cv_quality_chain = cv_quality_prompt | structured_llm

cv_quality_result = cv_quality_chain.invoke(
    {
        "cv_text": cv_text
    }
)

In [13]:
cv_quality_result

CVQualityAnalysis(overall_summary='The CV presents a strong candidate profile with relevant education and project experience in software engineering. However, it lacks clarity in the organization of skills and measurable results, which could enhance its impact.', scores=CVQualityScores(structure_and_readability_score=70, completeness_score=80, technical_skills_clarity_score=60, experience_description_score=75, projects_description_score=80, measurable_results_score=40, it_relevance_score=80), strengths=['Strong educational background with a high GPA in Software Engineering.', 'Relevant projects demonstrating practical application of skills in software development.', 'Clear specialization in Data Structures and Algorithms and System Programming.'], weaknesses=['Skills section lacks clear organization and grouping of technical skills.', 'No measurable results or quantifiable achievements are provided in project descriptions.'], missing_or_unclear_sections=['Specific details on internship

## 8. Convert Structured Result to Dictionary

In [14]:
if hasattr(cv_quality_result, "model_dump"):
    cv_quality_dict = cv_quality_result.model_dump()
else:
    cv_quality_dict = cv_quality_result.dict()


In [15]:
cv_quality_dict

{'overall_summary': 'The CV presents a strong candidate profile with relevant education and project experience in software engineering. However, it lacks clarity in the organization of skills and measurable results, which could enhance its impact.',
 'scores': {'structure_and_readability_score': 70,
  'completeness_score': 80,
  'technical_skills_clarity_score': 60,
  'experience_description_score': 75,
  'projects_description_score': 80,
  'measurable_results_score': 40,
  'it_relevance_score': 80},
 'strengths': ['Strong educational background with a high GPA in Software Engineering.',
  'Relevant projects demonstrating practical application of skills in software development.',
  'Clear specialization in Data Structures and Algorithms and System Programming.'],
 'weaknesses': ['Skills section lacks clear organization and grouping of technical skills.',
  'No measurable results or quantifiable achievements are provided in project descriptions.'],
 'missing_or_unclear_sections': ['Spec

## 9. Validate Individual Scores

In [16]:
def clamp_score(value):
    try:
        value = int(value)
    except (ValueError, TypeError):
        value = 0

    return max(0, min(100, value))


for score_name, score_value in cv_quality_dict["scores"].items():
    cv_quality_dict["scores"][score_name] = clamp_score(score_value)

In [17]:
cv_quality_dict["scores"]

{'structure_and_readability_score': 70,
 'completeness_score': 80,
 'technical_skills_clarity_score': 60,
 'experience_description_score': 75,
 'projects_description_score': 80,
 'measurable_results_score': 40,
 'it_relevance_score': 80}

## 10. Calculate Final CV Quality Score

In [18]:
score_weights = {
    "structure_and_readability_score": 0.15,
    "completeness_score": 0.15,
    "technical_skills_clarity_score": 0.20,
    "experience_description_score": 0.20,
    "projects_description_score": 0.15,
    "measurable_results_score": 0.10,
    "it_relevance_score": 0.05
}

final_cv_quality_score = 0


In [19]:
for score_name, weight in score_weights.items():
    final_cv_quality_score += cv_quality_dict["scores"][score_name] * weight

In [20]:
final_cv_quality_score = round(final_cv_quality_score)

In [21]:
final_cv_quality_score

70

## 11. Define CV Quality Category

In [22]:
def get_cv_quality_category(score):
    if score < 50:
        return "Weak CV"
    elif score < 70:
        return "Basic CV"
    elif score < 85:
        return "Good CV"
    else:
        return "Strong CV"


In [23]:
cv_quality_category = get_cv_quality_category(final_cv_quality_score)

In [24]:

cv_quality_category

'Good CV'

## 12. Add Final Score and Category to Result

In [25]:
cv_quality_dict["final_cv_quality_score"] = final_cv_quality_score
cv_quality_dict["cv_quality_category"] = cv_quality_category

In [26]:
cv_quality_dict

{'overall_summary': 'The CV presents a strong candidate profile with relevant education and project experience in software engineering. However, it lacks clarity in the organization of skills and measurable results, which could enhance its impact.',
 'scores': {'structure_and_readability_score': 70,
  'completeness_score': 80,
  'technical_skills_clarity_score': 60,
  'experience_description_score': 75,
  'projects_description_score': 80,
  'measurable_results_score': 40,
  'it_relevance_score': 80},
 'strengths': ['Strong educational background with a high GPA in Software Engineering.',
  'Relevant projects demonstrating practical application of skills in software development.',
  'Clear specialization in Data Structures and Algorithms and System Programming.'],
 'weaknesses': ['Skills section lacks clear organization and grouping of technical skills.',
  'No measurable results or quantifiable achievements are provided in project descriptions.'],
 'missing_or_unclear_sections': ['Spec

## 13. Create Score Breakdown Table

In [27]:
score_breakdown_df = pd.DataFrame([
    {
        "criterion": score_name,
        "score": score,
        "weight": score_weights.get(score_name, 0),
        "weighted_score": round(score * score_weights.get(score_name, 0), 2)
    }
    for score_name, score in cv_quality_dict["scores"].items()
])


In [28]:
score_breakdown_df

,criterion,score,weight,weighted_score
0,structure_and_readability_score,70,0.15,10.5
1,completeness_score,80,0.15,12.0
2,technical_skills_clarity_score,60,0.20,12.0
3,experience_description_score,75,0.20,15.0
4,projects_description_score,80,0.15,12.0
5,measurable_results_score,40,0.10,4.0
6,it_relevance_score,80,0.05,4.0


## 14 Generate Markdown Report

In [29]:
def format_bullet_list(items):

    if not items:
        return "- No items identified."

    return "\n".join([f"- {item}" for item in items])


cv_quality_markdown_report = f"""
# CV Quality Analysis Report

## Final CV Quality Score

**Score:** {cv_quality_dict["final_cv_quality_score"]}/100  
**Category:** {cv_quality_dict["cv_quality_category"]}

## Overall Summary

{cv_quality_dict["overall_summary"]}

## Score Breakdown

| Criterion | Score | Weight |
|---|---:|---:|
| Structure and readability | {cv_quality_dict["scores"]["structure_and_readability_score"]} | 15% |
| Completeness | {cv_quality_dict["scores"]["completeness_score"]} | 15% |
| Technical skills clarity | {cv_quality_dict["scores"]["technical_skills_clarity_score"]} | 20% |
| Experience description | {cv_quality_dict["scores"]["experience_description_score"]} | 20% |
| Projects description | {cv_quality_dict["scores"]["projects_description_score"]} | 15% |
| Measurable results | {cv_quality_dict["scores"]["measurable_results_score"]} | 10% |
| IT relevance | {cv_quality_dict["scores"]["it_relevance_score"]} | 5% |

## Strengths

{format_bullet_list(cv_quality_dict["strengths"])}

## Weaknesses

{format_bullet_list(cv_quality_dict["weaknesses"])}

## Missing or Unclear Sections

{format_bullet_list(cv_quality_dict["missing_or_unclear_sections"])}

## CV Improvement Recommendations

{format_bullet_list(cv_quality_dict["cv_improvement_recommendations"])}

## Evidence Notes

{format_bullet_list(cv_quality_dict["evidence_notes"])}
"""


In [30]:
print(cv_quality_markdown_report)


# CV Quality Analysis Report

## Final CV Quality Score

**Score:** 70/100  
**Category:** Good CV

## Overall Summary

The CV presents a strong candidate profile with relevant education and project experience in software engineering. However, it lacks clarity in the organization of skills and measurable results, which could enhance its impact.

## Score Breakdown

| Criterion | Score | Weight |
|---|---:|---:|
| Structure and readability | 70 | 15% |
| Completeness | 80 | 15% |
| Technical skills clarity | 60 | 20% |
| Experience description | 75 | 20% |
| Projects description | 80 | 15% |
| Measurable results | 40 | 10% |
| IT relevance | 80 | 5% |

## Strengths

- Strong educational background with a high GPA in Software Engineering.
- Relevant projects demonstrating practical application of skills in software development.
- Clear specialization in Data Structures and Algorithms and System Programming.

## Weaknesses

- Skills section lacks clear organization and grouping of techni

## 15. Save JSON Result & Markdown Report

In [31]:
with open(json_output_path, "w", encoding="utf-8") as file:
    json.dump(cv_quality_dict, file, indent=4, ensure_ascii=False)

with open(markdown_output_path, "w", encoding="utf-8") as file:
    file.write(cv_quality_markdown_report)

In [32]:
print(f"CV quality analysis JSON saved to: {json_output_path}")
print(f"CV quality Markdown report saved to: {markdown_output_path}")

CV quality analysis JSON saved to: ..\outputs\cv_quality\cv_quality_analysis.json
CV quality Markdown report saved to: ..\outputs\cv_quality\cv_quality_report.md
